- https://devblogs.microsoft.com/azure-sql/introducing-budget-bytes
    - https://github.com/Azure-Samples/budget-bytes-samples
- https://learn.microsoft.com/en-us/azure/azure-sql/database/free-offer
- https://devblogs.microsoft.com/semantic-kernel/build-ai-agents-with-github-copilot-sdk-and-microsoft-agent-framework
- https://developer.microsoft.com/blog/bringing-work-context-to-your-code-in-github-copilot (WorkIQ)
- 

# Observables & Rx.Net

Generate ASCII diagram how IObserver and IObservable work together in the Observer pattern.

```plaintext
+------------------------------+          +-----------------+
|   IObservable                |          |    IObserver    |
+------------------------------+          +-----------------+
| - observers: List<IObserver> |          | - Update(data)  |
| + Subscribe(observer)        |          +-----------------+
| + Unsubscribe(observer)      |
| + Notify(data)               |
+------------------------------+
        |                                         ^
        |                                         |
        v                                         |
+--------------------------+          +-------------------+
|   ConcreteObservable     |          |  ConcreteObserver |
+--------------------------+          +-------------------+
| - state: DataType        |          | - name: String    |
| + Subscribe(observer)    |          | + Update(data)    |
| + Unsubscribe(observer)  |          +-------------------+
| + Notify(data)           |
| + ChangeState(newState)  |
+--------------------------+
```

In [ ]:
class Unsubscriber<T> : IDisposable
{
    readonly List<IObserver<T>> observers;
    readonly IObserver<T> observer;

    public Unsubscriber(List<IObserver<T>> observers, IObserver<T> observer)
        => (this.observers, this.observer) = (observers, observer);

    public void Dispose()
    {
        if (observer != null && observers.Contains(observer))
            observers.Remove(observer);
    }
}

abstract class BaseObserable<T> : IObservable<T>
{
    protected readonly List<IObserver<T>> Observers = [];

    public IDisposable Subscribe(IObserver<T> observer)
    {
        if (!Observers.Contains(observer))
            Observers.Add(observer);

        return new Unsubscriber<T>(Observers, observer);
    }
}

In [ ]:
record TemperatureReading(string SensorId, double Temperature, DateTime Timestamp);

class TemperatureSensor : BaseObserable<TemperatureReading>
{
    readonly string sensorId;
    readonly Random random = new();

    public TemperatureSensor(string sensorId) => this.sensorId = sensorId;

    public void GenerateReading()
    {
        double temp = 20 + random.NextDouble() * 10; // Example: 20-30°C
        var reading = new TemperatureReading(sensorId, temp, DateTime.UtcNow);

        foreach (var observer in Observers)
            observer.OnNext(reading);
    }
}

In [ ]:
class TemperatureObserver : BaseObserable<double>, IObserver<TemperatureReading>
{
    readonly TimeSpan window = TimeSpan.FromSeconds(10);
    readonly List<TemperatureReading> readings = new();

    double? lastMax = null;

    public void OnNext(TemperatureReading value)
    {
        readings.Add(value);

        var cutoff = DateTime.UtcNow - window;
        readings.RemoveAll(r => r.Timestamp < cutoff);

        double currentMax = readings.Max(r => r.Temperature);

        if (lastMax == null || lastMax != currentMax)
        {
            foreach (var obs in Observers)
                obs.OnNext(currentMax);

            lastMax = currentMax;
        }
    }

    public void OnError(Exception error)
    {
        foreach (var obs in Observers)
            obs.OnError(error);
    }

    public void OnCompleted()
    {
        foreach (var obs in Observers)
            obs.OnCompleted();
    }
}

In [ ]:
int N = 3;
var sensors = new List<TemperatureSensor>();
for (int i = 0; i < N; i++)
    sensors.Add(new TemperatureSensor($"Sensor-{i + 1}"));

var maxObserver = new TemperatureObserver();
maxObserver.Subscribe(max =>
{
    if (!double.IsNaN(max))
        Console.WriteLine($"Max Temp: {max:F2}°C");
});

// Subscribe max observer to all sensors
foreach (var sensor in sensors)
    sensor.Subscribe(maxObserver);

// Simulate readings
var timer = new System.Timers.Timer(1000); // Every second
timer.Elapsed += (s, e) => sensors.ForEach(sensor => sensor.GenerateReading());
timer.Start();

await Task.Delay(TimeSpan.FromSeconds(30)); // Run for a while to see output

In [ ]:
#r "nuget: System.Reactive, 6.1.0"

In [ ]:
using System.Reactive.Linq;

int N = 3;
var random = new Random();
var sensors = new List<IObservable<TemperatureReading>>();

for (int i = 0; i < N; i++)
{
    string sensorId = $"Sensor-{i + 1}";

    // Each sensor emits a reading every second
    var sensorStream = Observable.Interval(TimeSpan.FromSeconds(1))
        .Select(_ => new TemperatureReading(sensorId, 20 + random.NextDouble() * 10, DateTime.UtcNow));

    sensors.Add(sensorStream);
}

var allSensors = sensors.Merge(); // from System.Reactive.Linq

// Sliding window max
var maxTempStream = allSensors
    .Window(TimeSpan.FromSeconds(10), TimeSpan.FromSeconds(1)) // sliding window
    .SelectMany(window => window
        .ToList() // collect readings in window
        .Select(list => list.Count > 0 ? list.Max(r => r.Temperature) : double.NaN))
    .Publish() // share single subscription
    .RefCount();

var changesOnly = maxTempStream.DistinctUntilChanged();
var periodic = Observable.Interval(TimeSpan.FromMinutes(1))
    .WithLatestFrom(maxTempStream, (_, max) => max);
var finalStream = changesOnly.Merge(periodic);

finalStream.Subscribe(max =>
{
    if (!double.IsNaN(max))
        Console.WriteLine($"Max Temp: {max:F2}°C");
});

await Task.Delay(TimeSpan.FromSeconds(30)); // Run for a while to see output

# Ix.Net Attempt

In [ ]:
//using System;
//using System.Collections.Generic;
//using System.Linq;
//using System.Threading.Tasks;
using System.Threading;
using System.Collections.Concurrent;
//using System.Interactive.Async;

async IAsyncEnumerable<TemperatureReading> CreateSensor(string sensorId, [System.Runtime.CompilerServices.EnumeratorCancellation] CancellationToken ct = default)
{
    while (!ct.IsCancellationRequested)
    {
        await Task.Delay(TimeSpan.FromSeconds(1), ct);
        yield return new TemperatureReading(sensorId, 20 + random.NextDouble() * 10, DateTime.UtcNow);
    }
}

async IAsyncEnumerable<TemperatureReading> MergeSensors(IEnumerable<IAsyncEnumerable<TemperatureReading>> sources, [System.Runtime.CompilerServices.EnumeratorCancellation] CancellationToken ct = default)
{
    var channel = new BlockingCollection<TemperatureReading>();

    var tasks = sources.Select(async src =>
    {
        await foreach (var item in src.WithCancellation(ct))
            channel.Add(item, ct);
    }).ToList();

    _ = Task.WhenAll(tasks).ContinueWith(_ => channel.CompleteAdding());

    foreach (var item in channel.GetConsumingEnumerable(ct))
        yield return item;
}

In [ ]:
record TemperatureReading(string SensorId, double Temperature, DateTime Timestamp);

In [ ]:
using System.Interactive.Async;

int N = 3;
var random = new Random();
var sensors = new List<IAsyncEnumerable<TemperatureReading>>();

for (int i = 0; i < N; i++)
{
    string sensorId = $"Sensor-{i + 1}";
    sensors.Add(CreateSensor(sensorId));
}

In [ ]:

using var cts = new CancellationTokenSource();

// Merge all sensors
var allSensors = MergeSensors(sensors, cts.Token);

// Sliding window (last 10 seconds, recomputed every 1 second)
var window = new List<TemperatureReading>();
double lastEmitted = double.NaN;

async Task RunAsync()
{
    await foreach (var reading in allSensors.WithCancellation(cts.Token))
    {
        var now = DateTime.UtcNow;

        // Add new reading
        window.Add(reading);

        // Remove old readings (>10s)
        window.RemoveAll(r => (now - r.Timestamp) > TimeSpan.FromSeconds(10));

        // Compute max
        double max = window.Count > 0 ? window.Max(r => r.Temperature) : double.NaN;

        // DistinctUntilChanged behavior
        if (!double.IsNaN(max) && max != lastEmitted)
        {
            Console.WriteLine($"Max Temp: {max:F2}°C");
            lastEmitted = max;
        }
    }
}

// Periodic (every 1 minute)
async Task RunPeriodicAsync()
{
    while (!cts.Token.IsCancellationRequested)
    {
        await Task.Delay(TimeSpan.FromMinutes(1), cts.Token);

        if (!double.IsNaN(lastEmitted))
            Console.WriteLine($"[Periodic] Max Temp: {lastEmitted:F2}°C");
    }
}

// Run both
var processingTask = RunAsync();
var periodicTask = RunPeriodicAsync();

// Let it run for a while
await Task.Delay(TimeSpan.FromSeconds(30));
cts.Cancel();

await Task.WhenAll(processingTask, periodicTask);